# GPU V1/V2/V3 Sanity Check: Mat32 / eps / top_q

本 notebook 用于测试你指定的配置组：

- `reg_lambda = 0.1`
- `N = 3_000_000`
- `dim = 2`
- `kernel = Mat32 (Matern nu=1.5)`
- `eps in [1e-5, 1e-7]`
- `top_q in [0, 32, 48, 64]`

说明：
- GPU V1 只覆盖 `top_q=0`（pure EFGP）。
- GPU V2 覆盖 `top_q>0`（CPU eigenspace + GPU preconditioner + GPU PCG）。
- GPU V3 覆盖 `top_q>0`（GPU eigenspace + GPU preconditioner + GPU PCG）。
- CPU 参考任务按 `top_q` 对齐：`top_q=0` 走 plain PCG；`top_q>0` 走 CPU eigenspace + CPU preconditioner + CPU PCG。
- 这里仍保留 CPU 参考任务；若规模过大可能超时或内存不足，会记录异常。
- 注意：若 `nufft_stage=cpu_finufft`，当前是 GPU solve + CPU NUFFT/precompute/predict 的混合收益；要看 full GPU，请确保 GPU NUFFT 可用。
- 本 notebook 直接打印结果，不写 CSV。

In [68]:
### For colab
'''
from google.colab import drive
import os

# 挂载云端硬盘
drive.mount('/content/drive')

# 切换到 ipynb 文件所在的具体文件夹
path = "/content/drive/My Drive/efgp_eigenpro_py/gpu/sanity_check"

if os.path.exists(path):
    os.chdir(path)
    print(f"当前工作目录已切换至: {os.getcwd()}")
    print("文件夹下的内容:")
    print(os.listdir())
else:
    print("错误：未找到路径，请检查 Drive 中的文件夹层级是否匹配。")

import os
import sys
from google.colab import drive

# 1. 挂载 Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. 切换到项目根目录
project_root = "/content/drive/MyDrive/efgp_eigenpro_py"
os.chdir(project_root)

# 3. 强行用 pip 安装 cufinufft 和相关 GPU 依赖
# 既然 conda 找不到，我们直接用 pip 安装 pre-built binaries
print("正在通过 pip 强制安装 cufinufft...")
!pip install cufinufft --extra-index-url https://pypi.nvidia.com

# 同时确保安装了处理 GPU 数组的核心库
!pip install cupy-cuda12x 

# 4. 安装项目 requirements.txt
print("\n正在同步项目依赖...")
!pip install -r requirements.txt

# 5. 切换到调试目录并添加路径
sanity_check_path = os.path.join(project_root, "gpu/sanity_check")
os.chdir(sanity_check_path)
if project_root not in sys.path:
    sys.path.append(project_root)

# 6. 环境最终验证
print("\n" + "="*40)
try:
    import torch
    import cufinufft
    import cupy
    print(f"✅ PyTorch: {torch.__version__} (GPU: {torch.cuda.is_available()})")
    print(f"✅ CuPy: {cupy.__version__}")
    print(f"✅ cufinufft 导入成功!")
    print(f"🚀 当前工作目录: {os.getcwd()}")
except Exception as e:
    print(f"❌ 环境依然报错: {e}")
print("="*40)
'''

'\nfrom google.colab import drive\nimport os\n\n# 挂载云端硬盘\ndrive.mount(\'/content/drive\')\n\n# 切换到 ipynb 文件所在的具体文件夹\npath = "/content/drive/My Drive/efgp_eigenpro_py/gpu/sanity_check"\n\nif os.path.exists(path):\n    os.chdir(path)\n    print(f"当前工作目录已切换至: {os.getcwd()}")\n    print("文件夹下的内容:")\n    print(os.listdir())\nelse:\n    print("错误：未找到路径，请检查 Drive 中的文件夹层级是否匹配。")\n\nimport os\nimport sys\nfrom google.colab import drive\n\n# 1. 挂载 Google Drive\nif not os.path.exists(\'/content/drive\'):\n    drive.mount(\'/content/drive\')\n\n# 2. 切换到项目根目录\nproject_root = "/content/drive/MyDrive/efgp_eigenpro_py"\nos.chdir(project_root)\n\n# 3. 强行用 pip 安装 cufinufft 和相关 GPU 依赖\n# 既然 conda 找不到，我们直接用 pip 安装 pre-built binaries\nprint("正在通过 pip 强制安装 cufinufft...")\n!pip install cufinufft --extra-index-url https://pypi.nvidia.com\n\n# 同时确保安装了处理 GPU 数组的核心库\n!pip install cupy-cuda12x \n\n# 4. 安装项目 requirements.txt\nprint("\n正在同步项目依赖...")\n!pip install -r requirements.txt\n\n# 5. 切换到调试目录并添加路径\nsanity_che

In [69]:
import os
import sys
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

# Ensure project root is importable no matter where notebook kernel starts.
# Notebook path: .../efgp_eigenpro_py/gpu/sanity_check/*.ipynb
# Needed sys.path entry: .../NU/ML
_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py import benchmark as bm
from efgp_eigenpro_py.benchmark import make_dataset, make_test_set, true_func_2d, compute_rmse
from efgp_eigenpro_py.discretization import GridSpec
from efgp_eigenpro_py.eigenspace import estimate_top_eigenspace
from efgp_eigenpro_py.eigenpro_precond import build_preconditioner
from efgp_eigenpro_py.gpu.backends import BackendConfig, build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import GPUOperatorContext, ensure_gpu_data_context
from efgp_eigenpro_py.gpu.iterative_solvers import pcg_solve_gpu
from efgp_eigenpro_py.gpu.v1_ops import apply_A_v1, predict_v1
from efgp_eigenpro_py.gpu.v2_preconditioner import GPUPreconditionerData, apply_preconditioner_v2
from efgp_eigenpro_py.gpu.versions import (
    GPURunConfig,
    run_v1_pure_efgp,
    run_v2_with_preconditioner_apply,
    run_v3_full_gpu_eigenspace,
)
from efgp_eigenpro_py.gpu.v3_eigenspace import EigenspaceConfig
from efgp_eigenpro_py.gpu import surrogate_ops as sur_ops

# optional: only needed to convert beta_gpu to host for diagnostics
try:
    import cupy as cp
except Exception:
    cp = None

np.set_printoptions(precision=6, suppress=True)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

cwd: d:\NU\ML\efgp_eigenpro_py\gpu\sanity_check
sys.path[0]: D:\NU\ML


In [70]:
# ----- user requested test matrix -----
REG_LAMBDA = 0.1
N_TRAIN = 10_000_000
DIM = 2
LENGTHSCALE = 0.1
NU = 1.5  # Mat32
EPS_LIST = [1e-6]
TOP_Q_LIST = [0, 16, 32, 48, 64]

N_TEST = 20_000
NOISE = 0.02
SEED_TRAIN = 0
SEED_TEST = 1

# GPU V1/V2/V3 solver controls
SOLVE_TOL = 1e-6
GPU_TOL = SOLVE_TOL
GPU_MAXITER = 3000
GPU_CHUNK_SIZE = None  # 可改成比如 200_000 做 chunk 预计算
GPU_DEBUG_FINITE_CHECKS = False
GPU_NUFFT = "auto"  # auto 或 cufinufft；用 cufinufft 可强制要求 GPU NUFFT

# Choose GPU versions to run, e.g. ["v2", "v3"]
GPU_VERSIONS = ['v1','v3']

# Align CPU/GPU grid selection
L2_SCALED = True

# V3 eigenspace controls
V3_OVERSAMPLE = 16
V3_N_ITER = 3

# CPU reference controls
RUN_CPU_REF = False
CPU_MAXITER = 3000
CPU_EIG_METHOD = "subspace_iter"
CPU_EIG_TOL = 1e-2
CPU_EIG_MAXITER = 20
CPU_EIG_BLOCK_SIZE = None
CPU_EIG_OVERSAMPLE = 2

# Surrogate eigenspace controls (GPU V3)
SURROGATE_ENABLED = []  # []/None 关闭; 可选: ["coarse_grid", "loose_eps", "subsample"]
SURROGATE_METHODS = {
    "coarse_grid": {
        "grid_scale": 0.35,
        "grid_m": None,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": V3_OVERSAMPLE,
        "block_size": None,
    },
    "loose_eps": {
        "eps_factor": 50.0,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": V3_OVERSAMPLE,
        "block_size": None,
    },
    "subsample": {
        "subsample_frac": 0.05,
        "subsample_seed": 0,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": V3_OVERSAMPLE,
        "block_size": None,
    },
}
COMBO_GRID_VALUES = {
    "grid_scale": [0.1],
    "eps_factor": [100],
    "subsample_frac": [0.3],
}
COMBO_COMMON_CFG = {
    "subsample_seed": 0,
    "sur_iter": 1,
    "refine_iter": 1,
    "oversample": V3_OVERSAMPLE,
    "block_size": None,
}
COMBO_ENABLED = [0,1]  # True/False；或 [0,1]（0=baseline,1=combo）；或 combo 名称列表


def _combo_tag(val: float) -> str:
    return str(val).replace(".", "p")


COMBO_VARIANTS = []
for grid_scale in COMBO_GRID_VALUES["grid_scale"]:
    for eps_factor in COMBO_GRID_VALUES["eps_factor"]:
        for subsample_frac in COMBO_GRID_VALUES["subsample_frac"]:
            combo_name = (
                f"combo_g{_combo_tag(grid_scale)}"
                f"_e{_combo_tag(eps_factor)}"
                f"_s{_combo_tag(subsample_frac)}"
            )
            combo_cfg = {
                "name": combo_name,
                "kind": "combo",
                "grid_scale": grid_scale,
                "grid_m": None,
                "eps_factor": eps_factor,
                "subsample_frac": subsample_frac,
                **COMBO_COMMON_CFG,
            }
            COMBO_VARIANTS.append(combo_cfg)

SURROGATE_VARIANTS = []
if SURROGATE_ENABLED is not None:
    for sur_name in SURROGATE_ENABLED:
        sur_cfg = SURROGATE_METHODS.get(sur_name)
        if sur_cfg is None:
            continue
        SURROGATE_VARIANTS.append({"name": sur_name, **sur_cfg})
V3_INCLUDE_BASELINE_WHEN_SURROGATE = False
_enable_combo = bool(COMBO_ENABLED)
_allowed_combo_names = None

if isinstance(COMBO_ENABLED, (list, tuple, set)):
    tokens = [str(v).strip() for v in COMBO_ENABLED]
    V3_INCLUDE_BASELINE_WHEN_SURROGATE = "0" in tokens
    has_combo_flag = "1" in tokens
    explicit_names = [t for t in tokens if t not in ("0", "1")]
    _enable_combo = has_combo_flag or len(explicit_names) > 0
    if len(explicit_names) > 0:
        _allowed_combo_names = set(explicit_names)

if _enable_combo:
    for combo_cfg in COMBO_VARIANTS:
        if _allowed_combo_names is not None and combo_cfg["name"] not in _allowed_combo_names:
            continue
        SURROGATE_VARIANTS.append(combo_cfg)

print("Run mode: print-only (no CSV output)")
print("GPU_VERSIONS:", GPU_VERSIONS)
print("L2_SCALED:", L2_SCALED)
print("RUN_CPU_REF:", RUN_CPU_REF)
print("SURROGATE_ENABLED:", SURROGATE_ENABLED)
print("COMBO_ENABLED:", COMBO_ENABLED)
print("COMBO_GRID_VALUES:", COMBO_GRID_VALUES)
print("COMBO_COMMON_CFG:", COMBO_COMMON_CFG)
print("V3_INCLUDE_BASELINE_WHEN_SURROGATE:", V3_INCLUDE_BASELINE_WHEN_SURROGATE)
print("SURROGATE_VARIANTS:", len(SURROGATE_VARIANTS))

Run mode: print-only (no CSV output)
GPU_VERSIONS: ['v1', 'v3']
L2_SCALED: True
RUN_CPU_REF: False
SURROGATE_ENABLED: []
COMBO_ENABLED: [0, 1]
COMBO_GRID_VALUES: {'grid_scale': [0.1], 'eps_factor': [100], 'subsample_frac': [0.3]}
COMBO_COMMON_CFG: {'subsample_seed': 0, 'sur_iter': 1, 'refine_iter': 1, 'oversample': 16, 'block_size': None}
V3_INCLUDE_BASELINE_WHEN_SURROGATE: True
SURROGATE_VARIANTS: 1


In [71]:
# 数据准备（请确认机器内存/GPU 显存足够）
print("Building dataset ...")
x_train, y_train = make_dataset(DIM, N_TRAIN, true_func_2d, noise=NOISE, seed=SEED_TRAIN)
x_test, y_test = make_test_set(DIM, N_TEST, true_func_2d, seed=SEED_TEST)

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)

kernel = make_matern(lengthscale=LENGTHSCALE, nu=NU, dim=DIM, variance=1.0)
print(kernel)

Building dataset ...
x_train: (10000000, 2) float64
y_train: (10000000,) float64
x_test : (20000, 2) float64
KernelSpec(fam='matern', dim=2, lengthscale=0.1, variance=1.0, nu=1.5)


In [72]:
def _mse_mae(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    diff = a - b
    mse = float(np.mean(np.abs(diff) ** 2))
    mae = float(np.mean(np.abs(diff)))
    return mse, mae


def _predict_gpu_yhat(cfg, state_cpu, beta_gpu, x_eval):
    backend_pred = build_gpu_backend_bundle(cfg.backend)
    dim = int(state_cpu.weights.ndim)
    dummy_x = np.zeros((1, dim), dtype=float)
    dummy_y = np.zeros((1,), dtype=float)
    data_ctx_pred = ensure_gpu_data_context(backend_pred, dummy_x, dummy_y, state_cpu)
    yhat_gpu = predict_v1(backend_pred, data_ctx_pred, x_eval, beta_gpu)
    if hasattr(yhat_gpu, "get"):
        return np.asarray(yhat_gpu.get())
    return np.asarray(yhat_gpu)


def run_gpu_v1_case(eps: float, cpu_beta=None) -> dict:
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=eps,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=GPU_CHUNK_SIZE,
        debug_finite_checks=GPU_DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    t0 = time.perf_counter()
    out = run_v1_pure_efgp(solver, x_train, y_train, cfg)
    t1 = time.perf_counter()

    # 用 CPU predict 做一个对齐质量指标（不影响 GPU V1 主流程）
    state_cpu = solver.precompute(x_train, y_train)
    m = int((state_cpu.grid.mtot - 1) // 2)
    if cp is None:
        beta_cpu = np.asarray(out.beta_gpu)
    else:
        beta_cpu = cp.asnumpy(out.beta_gpu)
    yhat = solver.predict(x_test, beta_cpu, state_cpu)
    rmse = compute_rmse(yhat, y_test)

    beta_mse = np.nan
    beta_mae = np.nan
    if cpu_beta is not None:
        beta_mse, beta_mae = _mse_mae(beta_cpu, cpu_beta)

    yhat_mse = np.nan
    yhat_mae = np.nan
    yhat_error = ""
    try:
        yhat_gpu_host = _predict_gpu_yhat(cfg, state_cpu, out.beta_gpu, x_test)
        yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat)
    except Exception as exc:
        yhat_mse = np.nan
        yhat_mae = np.nan
        yhat_error = f"{type(exc).__name__}: {exc}"

    row = {
        "mode": "gpu_v1_topq0",
        "eps": eps,
        "top_q": 0,
        "m": m,
        "rmse_test": float(rmse),
        "beta_mse": float(beta_mse),
        "beta_mae": float(beta_mae),
        "yhat_mse": float(yhat_mse),
        "yhat_mae": float(yhat_mae),
        "yhat_error": yhat_error,
        "wall_s_total": float(t1 - t0),
        "cg_iters": int(out.diagnostics.get("cg_iters", -1)),
        "cg_relres": float(out.diagnostics.get("cg_relres", np.nan)),
        "t_matvec_avg": float(out.diagnostics.get("t_matvec_avg", np.nan)),
        "t_matvec_total": float(out.diagnostics.get("t_matvec_total", np.nan)),
        "n_matvec": int(out.diagnostics.get("n_matvec", 0)),
        "time_precompute": float(out.diagnostics.get("time_precompute", np.nan)),
        "time_eigenspace": 0.0,
        "time_precond_build": 0.0,
        "eig_n_iter": 0,
        "eig_block_size": 0,
        "eig_residual_fro": np.nan,
        "eig_residual_fro_rel": np.nan,
        "time_solve": float(out.diagnostics.get("time_solve", np.nan)),
        "time_predict": float(out.diagnostics.get("time_predict", np.nan)),
        "nufft_stage": str(out.diagnostics.get("nufft_stage", "")),
        "device_name": str(out.diagnostics.get("device_name", "")),
        "n_precond": 0,
        "t_precond_total": 0.0,
        "status": "ok",
        "error": "",
    }
    return row


def run_gpu_v2_case(eps: float, top_q: int, cpu_beta=None) -> dict:
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=eps,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=GPU_CHUNK_SIZE,
        debug_finite_checks=GPU_DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    t0 = time.perf_counter()
    out = run_v2_with_preconditioner_apply(
        solver,
        x_train,
        y_train,
        cfg,
        top_q=int(top_q),
    )
    t1 = time.perf_counter()

    # 用 CPU predict 做一个对齐质量指标（不影响 GPU V2 主流程）
    state_cpu = solver.precompute(x_train, y_train)
    m = int((state_cpu.grid.mtot - 1) // 2)
    if cp is None:
        beta_cpu = np.asarray(out.beta_gpu)
    else:
        beta_cpu = cp.asnumpy(out.beta_gpu)
    yhat = solver.predict(x_test, beta_cpu, state_cpu)
    rmse = compute_rmse(yhat, y_test)

    beta_mse = np.nan
    beta_mae = np.nan
    if cpu_beta is not None:
        beta_mse, beta_mae = _mse_mae(beta_cpu, cpu_beta)

    yhat_mse = np.nan
    yhat_mae = np.nan
    yhat_error = ""
    try:
        yhat_gpu_host = _predict_gpu_yhat(cfg, state_cpu, out.beta_gpu, x_test)
        yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat)
    except Exception as exc:
        yhat_mse = np.nan
        yhat_mae = np.nan
        yhat_error = f"{type(exc).__name__}: {exc}"

    row = {
        "mode": "gpu_v2_topq",
        "eps": eps,
        "top_q": int(top_q),
        "m": m,
        "rmse_test": float(rmse),
        "beta_mse": float(beta_mse),
        "beta_mae": float(beta_mae),
        "yhat_mse": float(yhat_mse),
        "yhat_mae": float(yhat_mae),
        "yhat_error": yhat_error,
        "wall_s_total": float(t1 - t0),
        "cg_iters": int(out.diagnostics.get("cg_iters", -1)),
        "cg_relres": float(out.diagnostics.get("cg_relres", np.nan)),
        "t_matvec_avg": float(out.diagnostics.get("t_matvec_avg", np.nan)),
        "t_matvec_total": float(out.diagnostics.get("t_matvec_total", np.nan)),
        "n_matvec": int(out.diagnostics.get("n_matvec", 0)),
        "time_precompute": float(out.diagnostics.get("time_precompute", np.nan)),
        "time_eigenspace": float(out.diagnostics.get("time_eigenspace", np.nan)),
        "time_precond_build": float(out.diagnostics.get("time_precond_build", np.nan)),
        "eig_n_iter": int(out.diagnostics.get("eig_n_iter", 0)),
        "eig_block_size": int(out.diagnostics.get("eig_block_size", 0)),
        "eig_residual_fro": float(out.diagnostics.get("eig_residual_fro", np.nan)),
        "eig_residual_fro_rel": float(out.diagnostics.get("eig_residual_fro_rel", np.nan)),
        "time_solve": float(out.diagnostics.get("time_solve", np.nan)),
        "time_predict": float(out.diagnostics.get("time_predict", np.nan)),
        "nufft_stage": str(out.diagnostics.get("nufft_stage", "")),
        "device_name": str(out.diagnostics.get("device_name", "")),
        "n_precond": int(out.diagnostics.get("n_precond", 0)),
        "t_precond_total": float(out.diagnostics.get("t_precond_total", np.nan)),
        "status": "ok",
        "error": "",
    }
    return row


def run_gpu_v3_case(eps: float, top_q: int, cpu_beta=None) -> dict:
    if int(top_q) <= 0:
        raise ValueError("top_q must be > 0 for V3.")
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=eps,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=GPU_CHUNK_SIZE,
        debug_finite_checks=GPU_DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    block_size = int(top_q + V3_OVERSAMPLE)
    eig_cfg = EigenspaceConfig(q_max=int(top_q), block_size=block_size, n_iter=V3_N_ITER)

    t0 = time.perf_counter()
    out = run_v3_full_gpu_eigenspace(
        solver,
        x_train,
        y_train,
        cfg,
        eig_cfg,
    )
    t1 = time.perf_counter()

    # 用 CPU predict 做一个对齐质量指标（不影响 GPU V3 主流程）
    state_cpu = solver.precompute(x_train, y_train)
    m = int((state_cpu.grid.mtot - 1) // 2)
    if cp is None:
        beta_cpu = np.asarray(out.beta_gpu)
    else:
        beta_cpu = cp.asnumpy(out.beta_gpu)
    yhat = solver.predict(x_test, beta_cpu, state_cpu)
    rmse = compute_rmse(yhat, y_test)

    beta_mse = np.nan
    beta_mae = np.nan
    if cpu_beta is not None:
        beta_mse, beta_mae = _mse_mae(beta_cpu, cpu_beta)

    yhat_mse = np.nan
    yhat_mae = np.nan
    yhat_error = ""
    try:
        yhat_gpu_host = _predict_gpu_yhat(cfg, state_cpu, out.beta_gpu, x_test)
        yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat)
    except Exception as exc:
        yhat_mse = np.nan
        yhat_mae = np.nan
        yhat_error = f"{type(exc).__name__}: {exc}"

    eig_residual_fro_rel = out.diagnostics.get("eig_residual_fro_rel", np.nan)
    if not np.isfinite(eig_residual_fro_rel):
        cols_rel = out.diagnostics.get("eig_residual_cols_rel")
        if cols_rel is not None:
            eig_residual_fro_rel = float(np.linalg.norm(np.asarray(cols_rel)))

    row = {
        "mode": "gpu_v3_topq",
        "eps": eps,
        "top_q": int(top_q),
        "m": m,
        "rmse_test": float(rmse),
        "beta_mse": float(beta_mse),
        "beta_mae": float(beta_mae),
        "yhat_mse": float(yhat_mse),
        "yhat_mae": float(yhat_mae),
        "yhat_error": yhat_error,
        "wall_s_total": float(t1 - t0),
        "cg_iters": int(out.diagnostics.get("cg_iters", -1)),
        "cg_relres": float(out.diagnostics.get("cg_relres", np.nan)),
        "t_matvec_avg": float(out.diagnostics.get("t_matvec_avg", np.nan)),
        "t_matvec_total": float(out.diagnostics.get("t_matvec_total", np.nan)),
        "n_matvec": int(out.diagnostics.get("n_matvec", 0)),
        "time_precompute": float(out.diagnostics.get("time_precompute", np.nan)),
        "time_eigenspace": float(out.diagnostics.get("time_eigenspace", np.nan)),
        "time_precond_build": float(out.diagnostics.get("time_precond_build", np.nan)),
        "eig_n_iter": int(out.diagnostics.get("eig_n_iter", 0)),
        "eig_block_size": int(out.diagnostics.get("eig_block_size", 0)),
        "eig_residual_fro": float(out.diagnostics.get("eig_residual_fro", np.nan)),
        "eig_residual_fro_rel": float(eig_residual_fro_rel),
        "time_solve": float(out.diagnostics.get("time_solve", np.nan)),
        "time_predict": float(out.diagnostics.get("time_predict", np.nan)),
        "nufft_stage": str(out.diagnostics.get("nufft_stage", "")),
        "device_name": str(out.diagnostics.get("device_name", "")),
        "n_precond": int(out.diagnostics.get("n_precond", 0)),
        "t_precond_total": float(out.diagnostics.get("t_precond_total", np.nan)),
        "status": "ok",
        "error": "",
    }
    return row


def _build_surrogate_grid(fine_m: int, fine_h: float, grid_scale=None, grid_m=None) -> GridSpec:
    if grid_m is not None:
        m_coarse = int(grid_m)
    else:
        scale = 1.0 if grid_scale is None else float(grid_scale)
        m_coarse = max(1, int(fine_m * scale))
    xis = np.arange(-m_coarse, m_coarse + 1) * float(fine_h)
    return GridSpec(xis=xis, h=float(fine_h), mtot=xis.size, hm=m_coarse)


def _embed_coarse_to_fine(xp, vecs_coarse, coarse_m: int, fine_m: int, dim: int = 2):
    if coarse_m == fine_m:
        return xp.asarray(vecs_coarse)
    cols = []
    for j in range(vecs_coarse.shape[1]):
        v = xp.asarray(vecs_coarse[:, j]).reshape((2 * coarse_m + 1,) * dim)
        vf = xp.zeros((2 * fine_m + 1,) * dim, dtype=v.dtype)
        s = fine_m - coarse_m
        e = s + (2 * coarse_m + 1)
        if dim == 2:
            vf[s:e, s:e] = v
        else:
            vf[(slice(s, e),) * dim] = v
        cols.append(vf.reshape(-1))
    return xp.stack(cols, axis=1)


def run_gpu_v3_method_case(eps: float, top_q: int, cpu_beta=None, sur_cfg=None) -> dict:
    if sur_cfg is None:
        row = run_gpu_v3_case(eps, top_q, cpu_beta)
        row["eig_method"] = "subspace_iter"
        row["surrogate_tag"] = ""
        return row

    if int(top_q) <= 0:
        raise ValueError("top_q must be > 0 for V3.")

    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=eps,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=GPU_CHUNK_SIZE,
        debug_finite_checks=GPU_DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    backend = build_gpu_backend_bundle(cfg.backend)
    xp = backend.xp
    top_q = int(top_q)
    q_max = int(top_q)

    data_ctx = ensure_gpu_data_context(backend, x_train, y_train, state=None)
    data_ctx.meta["debug_finite_checks"] = bool(cfg.debug_finite_checks)
    op_ctx = GPUOperatorContext()

    t0 = time.perf_counter()
    data_ctx = sur_ops.gpu_precompute_v1(
        backend,
        solver.kernel,
        solver.eps,
        solver.nufft_tol,
        data_ctx,
        op_ctx,
        l2scaled=solver.l2scaled,
        chunk_size=cfg.chunk_size,
    )
    t1 = time.perf_counter()

    def _apply_block(local_ctx, local_op_ctx):
        def _fn(V):
            V = xp.asarray(V, dtype=xp.complex128)
            if V.ndim == 1:
                V = V.reshape(-1, 1)
            out = xp.empty_like(V)
            for i in range(V.shape[1]):
                apply_A_v1(
                    backend,
                    local_ctx,
                    V[:, i],
                    float(cfg.reg_lambda),
                    local_op_ctx,
                    out=out[:, i],
                )
            return out

        return _fn

    apply_A_fine = _apply_block(data_ctx, op_ctx)

    fine_mtot = int(data_ctx.meta.get("mtot", 0))
    fine_m = max(1, (fine_mtot - 1) // 2)
    fine_h = float(data_ctx.meta.get("h", 1.0))

    sur_kind = str(sur_cfg.get("kind", sur_cfg.get("name", "")))
    sur_name = str(sur_cfg.get("name", sur_kind))
    x_use = x_train
    y_use = y_train
    eps_sur = float(solver.eps)
    grid_override = None

    if sur_kind in ("subsample", "combo"):
        frac = float(sur_cfg.get("subsample_frac", 1.0))
        frac = min(max(frac, 1e-6), 1.0)
        n_sub = max(q_max + 2, int(len(x_train) * frac))
        n_sub = min(n_sub, len(x_train))
        seed = int(sur_cfg.get("subsample_seed", 0))
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(x_train), size=n_sub, replace=False)
        x_use = np.asarray(x_train[idx])
        y_use = np.asarray(y_train[idx])

    if sur_kind in ("loose_eps", "combo"):
        eps_factor = float(sur_cfg.get("eps_factor", 1.0))
        eps_sur = float(solver.eps) * eps_factor

    if sur_kind in ("coarse_grid", "combo"):
        grid_override = _build_surrogate_grid(
            fine_m,
            fine_h,
            grid_scale=sur_cfg.get("grid_scale", 1.0),
            grid_m=sur_cfg.get("grid_m", None),
        )

    data_ctx_sur = ensure_gpu_data_context(backend, x_use, y_use, state=None)
    data_ctx_sur.meta["debug_finite_checks"] = bool(cfg.debug_finite_checks)
    op_ctx_sur = GPUOperatorContext()

    t_sur_pre0 = time.perf_counter()
    data_ctx_sur = sur_ops.gpu_precompute_v1(
        backend,
        solver.kernel,
        eps_sur,
        solver.nufft_tol,
        data_ctx_sur,
        op_ctx_sur,
        l2scaled=solver.l2scaled,
        chunk_size=cfg.chunk_size,
        grid_override=grid_override,
    )
    t_sur_pre1 = time.perf_counter()

    size_sur = int(data_ctx_sur.rhs_gpu.size)
    if size_sur <= q_max:
        raise ValueError(f"surrogate size ({size_sur}) must be > top_q ({q_max})")

    sur_oversample = int(sur_cfg.get("oversample", V3_OVERSAMPLE))
    sur_block = sur_cfg.get("block_size")
    sur_block = int(q_max + sur_oversample) if sur_block is None else int(sur_block)
    sur_block = min(sur_block, size_sur)
    if sur_block <= q_max:
        sur_block = q_max + 1

    apply_A_sur = _apply_block(data_ctx_sur, op_ctx_sur)
    eig_cfg_sur = sur_ops.EigenspaceConfig(
        q_max=q_max,
        block_size=sur_block,
        n_iter=int(sur_cfg.get("sur_iter", 1)),
    )
    vals_sur, vecs_sur, _ = sur_ops.estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=apply_A_sur,
        size=size_sur,
        cfg=eig_cfg_sur,
    )

    init_Q = vecs_sur
    if grid_override is not None:
        init_Q = _embed_coarse_to_fine(xp, vecs_sur, int(grid_override.hm), fine_m, dim=DIM)

    eig_cfg_fine = sur_ops.EigenspaceConfig(
        q_max=q_max,
        block_size=max(sur_block, q_max + 1),
        n_iter=int(sur_cfg.get("refine_iter", 1)),
    )

    vals_gpu, vecs_gpu, eig_diag = sur_ops.estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=apply_A_fine,
        size=int(data_ctx.rhs_gpu.size),
        cfg=eig_cfg_fine,
        init_Q=init_Q,
    )
    t2 = time.perf_counter()

    if vals_gpu.size <= q_max:
        mu = float(vals_gpu[-1])
    else:
        mu = float(vals_gpu[q_max])
    scale_gpu = xp.asarray(1.0 - (mu / vals_gpu[:q_max]))
    precond_data = GPUPreconditionerData(
        U_gpu=vecs_gpu[:, :q_max],
        UH_gpu=vecs_gpu[:, :q_max].conj().T,
        scale_gpu=scale_gpu,
        scale_col_gpu=scale_gpu.reshape(-1, 1),
    )
    t3 = time.perf_counter()

    def _matvec(v, out):
        apply_A_v1(backend, data_ctx, v, float(cfg.reg_lambda), op_ctx, out=out)

    def _precond(v, out):
        apply_preconditioner_v2(backend, precond_data, v, op_ctx=op_ctx, out=out)

    beta_gpu, it, relres, stats = pcg_solve_gpu(
        backend,
        _matvec,
        _precond,
        data_ctx.rhs_gpu,
        op_ctx,
        cfg.tol,
        cfg.maxiter,
        return_stats=True,
    )
    t4 = time.perf_counter()

    _ = predict_v1(backend, data_ctx, x_train, beta_gpu)
    t5 = time.perf_counter()

    state_cpu = solver.precompute(x_train, y_train)
    m = int((state_cpu.grid.mtot - 1) // 2)
    if cp is None:
        beta_cpu = np.asarray(beta_gpu)
    else:
        beta_cpu = cp.asnumpy(beta_gpu)
    yhat = solver.predict(x_test, beta_cpu, state_cpu)
    rmse = compute_rmse(yhat, y_test)

    beta_mse = np.nan
    beta_mae = np.nan
    if cpu_beta is not None:
        beta_mse, beta_mae = _mse_mae(beta_cpu, cpu_beta)

    yhat_mse = np.nan
    yhat_mae = np.nan
    yhat_error = ""
    try:
        yhat_gpu_host = _predict_gpu_yhat(cfg, state_cpu, beta_gpu, x_test)
        yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat)
    except Exception as exc:
        yhat_error = f"{type(exc).__name__}: {exc}"

    row = {
        "mode": "gpu_v3_topq",
        "eig_method": f"sur_{sur_name}",
        "surrogate_tag": sur_name,
        "eps": eps,
        "top_q": int(top_q),
        "m": m,
        "rmse_test": float(rmse),
        "beta_mse": float(beta_mse),
        "beta_mae": float(beta_mae),
        "yhat_mse": float(yhat_mse),
        "yhat_mae": float(yhat_mae),
        "yhat_error": yhat_error,
        "wall_s_total": float(t5 - t0),
        "cg_iters": int(it),
        "cg_relres": float(relres),
        "t_matvec_avg": float(stats.get("t_matvec_avg", np.nan)),
        "t_matvec_total": float(stats.get("t_matvec_total", np.nan)),
        "n_matvec": int(stats.get("n_matvec", 0)),
        "time_precompute": float(t1 - t0),
        "time_eigenspace": float(t2 - t1),
        "time_precond_build": float(t3 - t2),
        "eig_n_iter": int(eig_diag.get("n_iter", 0)),
        "eig_block_size": int(eig_diag.get("block_size", 0)),
        "eig_residual_fro": float(eig_diag.get("residual_fro", np.nan)),
        "eig_residual_fro_rel": float(eig_diag.get("residual_fro_rel", np.nan)),
        "time_solve": float(t4 - t3),
        "time_predict": float(t5 - t4),
        "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
        "device_name": str(backend.device_name),
        "n_precond": int(stats.get("n_precond", 0)),
        "t_precond_total": float(stats.get("t_precond_total", np.nan)),
        "surrogate_precompute": float(t_sur_pre1 - t_sur_pre0),
        "status": "ok",
        "error": "",
    }
    return row


def run_cpu_case(eps: float, top_q: int) -> dict:
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=eps,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )

    t0 = time.perf_counter()
    state = solver.precompute(x_train, y_train)
    t1 = time.perf_counter()

    matvec = lambda v: solver._apply_A(state, v)

    # CPU reference aligned with top_q (top_q=0 plain, top_q>0 with precond)
    precond_apply = None
    top_q_used = int(top_q)
    t_eig = 0.0
    t_precond_build = 0.0
    eig_n_iter = 0
    eig_block_size = 0
    eig_residual_fro = np.nan

    if top_q_used > 0:
        t_e0 = time.perf_counter()
        eigpairs = estimate_top_eigenspace(
            matvec,
            size=state.rhs.size,
            top_q=top_q_used,
            method=CPU_EIG_METHOD,
            tol=CPU_EIG_TOL,
            maxiter=CPU_EIG_MAXITER,
            matvec_block=lambda V: solver._apply_A_block(state, V),
            block_size=CPU_EIG_BLOCK_SIZE,
            oversample=CPU_EIG_OVERSAMPLE,
        )
        t_e1 = time.perf_counter()

        if eigpairs.values.size > top_q_used:
            mu = float(eigpairs.values[top_q_used])
        else:
            mu = float(eigpairs.values[-1])

        t_p0 = time.perf_counter()
        precond = build_preconditioner(
            eigpairs.values[:top_q_used],
            eigpairs.vectors[:, :top_q_used],
            mu,
        )
        t_p1 = time.perf_counter()

        precond_apply = precond.apply
        t_eig = t_e1 - t_e0
        t_precond_build = t_p1 - t_p0
        eig_n_iter = int(CPU_EIG_MAXITER) if CPU_EIG_MAXITER is not None else 0
        eig_block_size = 0 if CPU_EIG_BLOCK_SIZE is None else int(CPU_EIG_BLOCK_SIZE)

    t_s0 = time.perf_counter()
    beta, it, relres, _hist, stats = bm.pcg_solve(
        matvec,
        state.rhs,
        tol=SOLVE_TOL,
        maxiter=CPU_MAXITER,
        precond=precond_apply,
        store_history=False,
        return_stats=True,
    )
    n_precond = int(stats.get("n_precond", 0))
    t_precond_total = float(stats.get("t_precond_total", 0.0))
    t_s1 = time.perf_counter()

    t_p0 = time.perf_counter()
    yhat = solver.predict(x_test, beta, state)
    t_p1 = time.perf_counter()

    m = int((state.grid.mtot - 1) // 2)
    row = {
        "mode": "cpu_ref",
        "eps": eps,
        "top_q": int(top_q_used),
        "m": m,
        "rmse_test": float(compute_rmse(yhat, y_test)),
        "beta_mse": np.nan,
        "beta_mae": np.nan,
        "yhat_mse": np.nan,
        "yhat_mae": np.nan,
        "wall_s_total": float((t1 - t0) + t_eig + t_precond_build + (t_s1 - t_s0) + (t_p1 - t_p0)),
        "cg_iters": int(it),
        "cg_relres": float(relres),
        "t_matvec_avg": float(stats["t_matvec_total"] / max(stats["n_matvec"], 1)),
        "t_matvec_total": float(stats["t_matvec_total"]),
        "n_matvec": int(stats["n_matvec"]),
        "time_precompute": float(t1 - t0),
        "time_eigenspace": float(t_eig),
        "time_precond_build": float(t_precond_build),
        "eig_n_iter": int(eig_n_iter),
        "eig_block_size": int(eig_block_size),
        "eig_residual_fro": float(eig_residual_fro),
        "eig_residual_fro_rel": np.nan,
        "time_solve": float(t_s1 - t_s0),
        "time_predict": float(t_p1 - t_p0),
        "nufft_stage": "cpu_finufft",
        "device_name": "cpu",
        "n_precond": int(n_precond),
        "t_precond_total": float(t_precond_total),
        "status": "ok",
        "error": "",
    }
    return row, beta

In [73]:
rows = []
run_all = "all" in GPU_VERSIONS

for eps in EPS_LIST:
    for q in TOP_Q_LIST:
        q = int(q)
        cpu_beta = None

        if RUN_CPU_REF:
            try:
                cpu_row, cpu_beta = run_cpu_case(eps, q)
                rows.append(cpu_row)
            except Exception as exc:
                tb = traceback.format_exc()
                print(f"\n[CPU ERROR] eps={eps}, top_q={q}")
                print(tb)
                rows.append(
                    {
                        "mode": "cpu_ref",
                        "eps": eps,
                        "top_q": q,
                        "rmse_test": np.nan,
                        "beta_mse": np.nan,
                        "beta_mae": np.nan,
                        "wall_s_total": np.nan,
                        "cg_iters": None,
                        "cg_relres": np.nan,
                        "t_matvec_avg": np.nan,
                        "t_matvec_total": np.nan,
                        "n_matvec": np.nan,
                        "time_precompute": np.nan,
                        "time_eigenspace": np.nan,
                        "time_precond_build": np.nan,
                        "eig_n_iter": np.nan,
                        "eig_block_size": np.nan,
                        "eig_residual_fro": np.nan,
                        "eig_residual_fro_rel": np.nan,
                        "time_solve": np.nan,
                        "time_predict": np.nan,
                        "nufft_stage": "cpu_finufft",
                        "device_name": "",
                        "n_precond": np.nan,
                        "t_precond_total": np.nan,
                        "status": "error",
                        "error": tb,
                    }
                )
                cpu_beta = None

        if q == 0 and (run_all or "v1" in GPU_VERSIONS):
            try:
                rows.append(run_gpu_v1_case(eps, cpu_beta))
            except Exception as exc:
                tb = traceback.format_exc()
                print(f"\n[GPU V1 ERROR] eps={eps}")
                print(tb)
                rows.append(
                    {
                        "mode": "gpu_v1_topq0",
                        "eps": eps,
                        "top_q": 0,
                        "rmse_test": np.nan,
                        "beta_mse": np.nan,
                        "beta_mae": np.nan,
                        "wall_s_total": np.nan,
                        "cg_iters": None,
                        "cg_relres": np.nan,
                        "t_matvec_avg": np.nan,
                        "t_matvec_total": np.nan,
                        "n_matvec": np.nan,
                        "time_precompute": np.nan,
                        "time_eigenspace": np.nan,
                        "time_precond_build": np.nan,
                        "eig_n_iter": np.nan,
                        "eig_block_size": np.nan,
                        "eig_residual_fro": np.nan,
                        "eig_residual_fro_rel": np.nan,
                        "time_solve": np.nan,
                        "time_predict": np.nan,
                        "nufft_stage": "",
                        "device_name": "",
                        "n_precond": np.nan,
                        "t_precond_total": np.nan,
                        "status": "error",
                        "error": tb,
                    }
                )

        if q > 0 and (run_all or "v2" in GPU_VERSIONS):
            try:
                rows.append(run_gpu_v2_case(eps, q, cpu_beta))
            except Exception as exc:
                tb = traceback.format_exc()
                print(f"\n[GPU V2 ERROR] eps={eps}, top_q={q}")
                print(tb)
                rows.append(
                    {
                        "mode": "gpu_v2_topq",
                        "eps": eps,
                        "top_q": q,
                        "rmse_test": np.nan,
                        "beta_mse": np.nan,
                        "beta_mae": np.nan,
                        "wall_s_total": np.nan,
                        "cg_iters": None,
                        "cg_relres": np.nan,
                        "t_matvec_avg": np.nan,
                        "t_matvec_total": np.nan,
                        "n_matvec": np.nan,
                        "time_precompute": np.nan,
                        "time_eigenspace": np.nan,
                        "time_precond_build": np.nan,
                        "eig_n_iter": np.nan,
                        "eig_block_size": np.nan,
                        "eig_residual_fro": np.nan,
                        "eig_residual_fro_rel": np.nan,
                        "time_solve": np.nan,
                        "time_predict": np.nan,
                        "nufft_stage": "",
                        "device_name": "",
                        "n_precond": np.nan,
                        "t_precond_total": np.nan,
                        "status": "error",
                        "error": tb,
                    }
                )

        if q > 0 and (run_all or "v3" in GPU_VERSIONS):
            v3_method_cfgs = []
            if V3_INCLUDE_BASELINE_WHEN_SURROGATE or not SURROGATE_VARIANTS:
                v3_method_cfgs.append(None)
            v3_method_cfgs.extend(list(SURROGATE_VARIANTS))
            for sur_cfg in v3_method_cfgs:
                method_name = "subspace_iter" if sur_cfg is None else f"sur_{sur_cfg.get('name', 'unknown')}"
                try:
                    rows.append(run_gpu_v3_method_case(eps, q, cpu_beta, sur_cfg=sur_cfg))
                except Exception as exc:
                    tb = traceback.format_exc()
                    print(f"\n[GPU V3 ERROR] eps={eps}, top_q={q}, method={method_name}")
                    print(tb)
                    rows.append(
                        {
                            "mode": "gpu_v3_topq",
                            "eig_method": method_name,
                            "surrogate_tag": "" if sur_cfg is None else str(sur_cfg.get("name", "")),
                            "eps": eps,
                            "top_q": q,
                            "rmse_test": np.nan,
                            "beta_mse": np.nan,
                            "beta_mae": np.nan,
                            "wall_s_total": np.nan,
                            "cg_iters": None,
                            "cg_relres": np.nan,
                            "t_matvec_avg": np.nan,
                            "t_matvec_total": np.nan,
                            "n_matvec": np.nan,
                            "time_precompute": np.nan,
                            "time_eigenspace": np.nan,
                            "time_precond_build": np.nan,
                            "eig_n_iter": np.nan,
                            "eig_block_size": np.nan,
                            "eig_residual_fro": np.nan,
                            "eig_residual_fro_rel": np.nan,
                            "time_solve": np.nan,
                            "time_predict": np.nan,
                            "nufft_stage": "",
                            "device_name": "",
                            "n_precond": np.nan,
                            "t_precond_total": np.nan,
                            "status": "error",
                            "error": tb,
                        }
                    )


df = pd.DataFrame(rows)
print("Rows:", len(df))
df

Rows: 9


,mode,eps,top_q,m,rmse_test,beta_mse,beta_mae,yhat_mse,yhat_mae,yhat_error,...,time_predict,nufft_stage,device_name,n_precond,t_precond_total,status,error,eig_method,surrogate_tag,surrogate_precompute
0,gpu_v1_topq0,0.000001,0,181,0.000641,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.588371,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,0,0.000000,ok,,NaN,NaN,NaN
1,gpu_v3_topq,0.000001,16,181,0.000643,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.584292,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,1216,3.588215,ok,,subspace_iter,,NaN
2,gpu_v3_topq,0.000001,16,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.679752,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,1186,3.218247,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.424385
3,gpu_v3_topq,0.000001,32,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.578882,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,876,2.757417,ok,,subspace_iter,,NaN
4,gpu_v3_topq,0.000001,32,181,0.000645,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.637489,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,860,2.639864,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.359718
5,gpu_v3_topq,0.000001,48,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.601084,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,664,2.386716,ok,,subspace_iter,,NaN
6,gpu_v3_topq,0.000001,48,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.631674,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,694,2.535861,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.369708
7,gpu_v3_topq,0.000001,64,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.644873,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,557,2.225173,ok,,subspace_iter,,NaN
8,gpu_v3_topq,0.000001,64,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,...,0.622044,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,572,1.961370,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.356721


In [74]:
import pandas as pd
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(df)

,mode,eps,top_q,m,rmse_test,beta_mse,beta_mae,yhat_mse,yhat_mae,yhat_error,wall_s_total,cg_iters,cg_relres,t_matvec_avg,t_matvec_total,n_matvec,time_precompute,time_eigenspace,time_precond_build,eig_n_iter,eig_block_size,eig_residual_fro,eig_residual_fro_rel,time_solve,time_predict,nufft_stage,device_name,n_precond,t_precond_total,status,error,eig_method,surrogate_tag,surrogate_precompute
0,gpu_v1_topq0,0.000001,0,181,0.000641,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,13.206427,1761,8.532344e-07,0.005290,9.320472,1762,1.749769,0.000000,0.000000,0,0,NaN,NaN,10.826185,0.588371,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,0,0.000000,ok,,NaN,NaN,NaN
1,gpu_v3_topq,0.000001,16,181,0.000643,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,14.757556,1216,9.709605e-07,0.005297,6.446393,1217,1.470865,1.067700,0.000277,3,32,23753.751487,NaN,11.581911,0.584292,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,1216,3.588215,ok,,subspace_iter,,NaN
2,gpu_v3_topq,0.000001,16,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,14.136285,1186,8.435535e-07,0.004812,5.711486,1187,1.496261,1.621405,0.000254,1,32,53872.451114,0.040631,10.338612,0.679752,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,1186,3.218247,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.424385
3,gpu_v3_topq,0.000001,32,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,12.714739,876,9.505463e-07,0.005217,4.575121,877,1.523524,2.174691,0.000262,3,48,17295.391937,NaN,8.413309,0.578882,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,876,2.757417,ok,,subspace_iter,,NaN
4,gpu_v3_topq,0.000001,32,181,0.000645,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,13.090119,860,8.882048e-07,0.005507,4.741149,861,1.458100,2.302077,0.000257,1,48,45677.909086,0.032332,8.692195,0.637489,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,860,2.639864,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.359718
5,gpu_v3_topq,0.000001,48,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,12.016287,664,9.181842e-07,0.005638,3.749319,665,1.521265,2.866935,0.000286,3,64,13515.904858,NaN,7.006789,0.601084,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,664,2.386716,ok,,subspace_iter,,NaN
6,gpu_v3_topq,0.000001,48,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,12.489546,694,9.643480e-07,0.006026,4.188080,695,1.454884,2.743440,0.000318,1,64,31490.388426,0.021958,7.659230,0.631674,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,694,2.535861,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.369708
7,gpu_v3_topq,0.000001,64,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,13.843596,557,8.145813e-07,0.005572,3.108972,558,1.462880,5.647923,0.000331,3,80,10773.750313,NaN,6.037661,0.644873,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,557,2.225173,ok,,subspace_iter,,NaN
8,gpu_v3_topq,0.000001,64,181,0.000644,NaN,NaN,NaN,NaN,ValueError: x_eval.shape[1] must match precomp...,17.736761,572,9.735880e-07,0.004426,2.535974,573,1.443835,10.532152,0.000246,1,80,24164.334580,0.016766,5.138485,0.622044,cpu_finufft,NVIDIA GeForce RTX 3050 Laptop GPU,572,1.961370,ok,,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.356721


In [75]:
# 便于快速查看每组 (eps, mode, top_q) 结果
cols = [
    "mode", "eig_method", "surrogate_tag", "eps", "top_q", "m", "status", "rmse_test", "cg_iters", "cg_relres",
    "n_matvec", "t_matvec_avg", "t_matvec_total", "n_precond", "t_precond_total",
    "time_precompute", "time_eigenspace", "time_precond_build",
    "eig_n_iter", "eig_block_size", "eig_residual_fro",
    "time_solve", "time_predict", "wall_s_total", "error",
]

for col in cols:
    if col not in df.columns:
        df[col] = np.nan

display(df[cols].sort_values(["eps", "mode", "top_q"]))

,mode,eig_method,surrogate_tag,eps,top_q,m,status,rmse_test,cg_iters,cg_relres,...,time_precompute,time_eigenspace,time_precond_build,eig_n_iter,eig_block_size,eig_residual_fro,time_solve,time_predict,wall_s_total,error
0,gpu_v1_topq0,NaN,NaN,0.000001,0,181,ok,0.000641,1761,8.532344e-07,...,1.749769,0.000000,0.000000,0,0,NaN,10.826185,0.588371,13.206427,
1,gpu_v3_topq,subspace_iter,,0.000001,16,181,ok,0.000643,1216,9.709605e-07,...,1.470865,1.067700,0.000277,3,32,23753.751487,11.581911,0.584292,14.757556,
2,gpu_v3_topq,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.000001,16,181,ok,0.000644,1186,8.435535e-07,...,1.496261,1.621405,0.000254,1,32,53872.451114,10.338612,0.679752,14.136285,
3,gpu_v3_topq,subspace_iter,,0.000001,32,181,ok,0.000644,876,9.505463e-07,...,1.523524,2.174691,0.000262,3,48,17295.391937,8.413309,0.578882,12.714739,
4,gpu_v3_topq,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.000001,32,181,ok,0.000645,860,8.882048e-07,...,1.458100,2.302077,0.000257,1,48,45677.909086,8.692195,0.637489,13.090119,
5,gpu_v3_topq,subspace_iter,,0.000001,48,181,ok,0.000644,664,9.181842e-07,...,1.521265,2.866935,0.000286,3,64,13515.904858,7.006789,0.601084,12.016287,
6,gpu_v3_topq,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.000001,48,181,ok,0.000644,694,9.643480e-07,...,1.454884,2.743440,0.000318,1,64,31490.388426,7.659230,0.631674,12.489546,
7,gpu_v3_topq,subspace_iter,,0.000001,64,181,ok,0.000644,557,8.145813e-07,...,1.462880,5.647923,0.000331,3,80,10773.750313,6.037661,0.644873,13.843596,
8,gpu_v3_topq,sur_combo_g0p1_e100_s0p3,combo_g0p1_e100_s0p3,0.000001,64,181,ok,0.000644,572,9.735880e-07,...,1.443835,10.532152,0.000246,1,80,24164.334580,5.138485,0.622044,17.736761,
